In [ ]:
import pandas as pd

data=pd.read_csv('IndianElection19TwitterData.csv')
print(data.shape)
print(data['User'].nunique())

data['Date']=pd.to_datetime(data['Date'])
data.drop(columns=['Unnamed: 0'], inplace=True)

data.head()

(142566, 4)
49397
Duplicate Tweets: 0


,Date,User,Tweet
0,2019-05-18 23:50:47+00:00,advosushildixit,@anjanaomkashyap I am seeing you as future #bj...
1,2019-05-18 23:20:00+00:00,airnewsalerts,Trinamool Congress Sitting MP Abhishek Banerje...
2,2019-05-18 23:00:03+00:00,jiaeur,#LokSabhaElections2019 \n23rd May 2019 will re...
3,2019-05-18 22:53:54+00:00,PVenkatGandhi,#LokSabhaElections2019 \n23rd May 2019 will re...
4,2019-05-18 22:20:48+00:00,TheNirbhay1,PM Modi creates a new record of being the only...


In [ ]:
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):

    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()

    words = [word for word in words if word not in stop_words and len(word) > 2]

    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

data['Clean_Tweet'] = data['Tweet'].apply(clean_text)

data[['Tweet','Clean_Tweet']].head(10)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,Tweet,Clean_Tweet
0,@anjanaomkashyap I am seeing you as future #bj...,seeing future bjp spokesperson good luck anjan...
1,Trinamool Congress Sitting MP Abhishek Banerje...,trinamool congress sitting abhishek banerjee s...
2,#LokSabhaElections2019 \n23rd May 2019 will re...,loksabhaelections may reveal even ecisveep cou...
3,#LokSabhaElections2019 \n23rd May 2019 will re...,loksabhaelections may reveal even could help m...
4,PM Modi creates a new record of being the only...,modi creates new record democratic country con...
5,My somewhat biased exit poll for India electio...,somewhat biased exit poll india election based...
6,@rupasubramanya Even assuming statistical erro...,even assuming statistical error estimation bjp...
7,@abhijitmajumder Small correction. Nobody gets...,small correction nobody get appointed bjp work...
8,We still fucking dancing 🕺🏼 ♏️ #INC,still fucking dancing inc
9,@abhijitmajumder Appointment of Successor! \n\...,appointment successor god forbid allow bjp sto...


In [ ]:
data.to_csv("cleaned_election_tweets.csv", index=False)

,Date,User,Tweet,Clean_Tweet
0,2019-05-18 23:50:47+00:00,advosushildixit,@anjanaomkashyap I am seeing you as future #bj...,seeing future bjp spokesperson good luck anjan...
1,2019-05-18 23:20:00+00:00,airnewsalerts,Trinamool Congress Sitting MP Abhishek Banerje...,trinamool congress sitting abhishek banerjee s...
2,2019-05-18 23:00:03+00:00,jiaeur,#LokSabhaElections2019 \n23rd May 2019 will re...,loksabhaelections may reveal even ecisveep cou...
3,2019-05-18 22:53:54+00:00,PVenkatGandhi,#LokSabhaElections2019 \n23rd May 2019 will re...,loksabhaelections may reveal even could help m...
4,2019-05-18 22:20:48+00:00,TheNirbhay1,PM Modi creates a new record of being the only...,modi creates new record democratic country con...


In [ ]:
!pip install vaderSentiment


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.9 MB/s eta 0:00:00


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

def get_sentiment_scores(text):
    score = analyzer.polarity_scores(text)

    return pd.Series([
        score['neg'],
        score['neu'],
        score['pos'],
        score['compound']
    ])


data[['Negative', 'Neutral', 'Positive', 'Compound']] = data['Clean_Tweet'].apply(get_sentiment_scores)


In [ ]:
def sentiment_label(compound):

    if compound >= 0.05:
        return "Positive"

    elif compound <= -0.05:
        return "Negative"

    else:
        return "Neutral"

data['Sentiment'] = data['Compound'].apply(sentiment_label)

In [ ]:
data['Sentiment'].value_counts()
sentiment_percentage = round(
    data['Sentiment'].value_counts(normalize=True)*100,
    2
)

print(sentiment_percentage)

Sentiment
Positive    49.52
Negative    28.09
Neutral     22.39
Name: proportion, dtype: float64


In [ ]:
data.head()

,Date,User,Tweet,Clean_Tweet,Negative,Neutral,Positive,Compound,Sentiment
0,2019-05-18 23:50:47+00:00,advosushildixit,@anjanaomkashyap I am seeing you as future #bj...,seeing future bjp spokesperson good luck anjan...,0.0,0.656,0.344,0.8126,Positive
1,2019-05-18 23:20:00+00:00,airnewsalerts,Trinamool Congress Sitting MP Abhishek Banerje...,trinamool congress sitting abhishek banerjee s...,0.0,1.000,0.000,0.0000,Neutral
2,2019-05-18 23:00:03+00:00,jiaeur,#LokSabhaElections2019 \n23rd May 2019 will re...,loksabhaelections may reveal even ecisveep cou...,0.0,0.711,0.289,0.7579,Positive
3,2019-05-18 22:53:54+00:00,PVenkatGandhi,#LokSabhaElections2019 \n23rd May 2019 will re...,loksabhaelections may reveal even could help m...,0.0,0.698,0.302,0.7579,Positive
4,2019-05-18 22:20:48+00:00,TheNirbhay1,PM Modi creates a new record of being the only...,modi creates new record democratic country con...,0.0,0.663,0.337,0.9217,Positive
